# 🏠 Aerial House Detection — Research-Backed Implementation

**Pipeline:** WHU Building Dataset (aerial, CC BY 4.0 via mirror) → COCO/mask → YOLO bbox → **YOLO11s** (Ultralytics) → **SAHI** sliced inference for high-res → **house counting** (unique detections after merge).

**Decisions from Phase-1 research (do not change without technical reason):**
- Dataset: WHU Building aerial subset — 8,189 tiles 512×512 @0.3 m, official split 4,736/1,036/2,416
- Model: YOLO11s primary, YOLO26s secondary comparison
- Small objects: SAHI slicing (slice 512–640 px, overlap 0.2)
- Metrics: mAP@0.5, mAP@0.5:0.95, P/R/F1 + **count MAE / % error** (differentiator)
- Video: later phase (BoT-SORT, unique track IDs)

In [ ]:
# Phase 1 — Environment check: GPU, CUDA, disk
import subprocess, os, sys, shutil

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python:", sys.version.split()[0])
print("GPU:", sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "NO GPU — enable Runtime > Change runtime type > T4 GPU")
print("Disk free (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
print("RAM GB:", round(float(sh("free -g | awk 'NR==2 {print $2}'")) or 0, 1))
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Phase 2 — Dataset Acquisition

**Source (verified 2026-11-09):** Hugging Face mirror `giswqs/WHU-Building-Dataset` (license **CC BY 4.0** per official dataset card).
Original: WHU Building Dataset, Wuhan University GPCV — Ji et al., IEEE TGRS 2019 (https://gpcv.whu.edu.cn/data/building_dataset.html).

**Format found:** raster binary masks (0/255 PNG), NOT polygons → conversion = **connected-component labeling → per-instance bbox** (validated geometrically in the conversion test cell).
**Split used (fixed, no reshuffling):** train 5,732 / val 1,228 / test 1,228 tiles, 512×512 — tile-level split, no geographic leakage between splits.
⚠️ Known limitation (documented): touching buildings merge into one mask component → one bbox. Counting uses the COCO-style instance separation where possible; merged roofs are flagged in the error analysis.

In [ ]:
# Phase 2 — Install dependencies (Ultralytics YOLO11/26 + SAHI + pycocotools)
import subprocess
r = subprocess.run(
    ["pip", "install", "-q", "ultralytics", "sahi", "pycocotools"],
    capture_output=True, text=True,
)
print("pip exit:", r.returncode)
if r.returncode != 0:
    print(r.stderr[-2000:])

In [ ]:
# Phase 2 — Verify installs + download WHU dataset from HF mirror (runs in background thread)
import subprocess, threading, time, os

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# verify versions (import check)
r = sh("python -c \"import ultralytics, sahi; print('ultralytics', ultralytics.__version__); print('sahi', sahi.__version__)\"")
print(r.stdout or r.stderr)

# download in background — snapshot_download resumes if interrupted
os.makedirs("/content/aerial-house-detection", exist_ok=True)
DL = r'''
from huggingface_hub import snapshot_download
p = snapshot_download(repo_id="giswqs/WHU-Building-Dataset", repo_type="dataset",
                      local_dir="/content/aerial-house-detection/datasets/whu")
print("DONE", p)
'''
with open("/content/dl_whu.py", "w") as f:
    f.write(DL)

def run_download():
    r = sh("python /content/dl_whu.py > /content/dl_whu.log 2>&1")
    with open("/content/dl_done.flag", "w") as f:
        f.write("ok" if r.returncode == 0 else f"FAIL {r.returncode}")

th = threading.Thread(target=run_download, daemon=True)
th.start()
print("Download started in background. Log: /content/dl_whu.log (check next cell)")

In [ ]:
# Phase 2 — Compact download status (re-run)
import os, glob, re
log = "/content/dl_whu.log"
if os.path.exists("/content/dl_done.flag"):
    print("FLAG:", open("/content/dl_done.flag").read())
if os.path.exists(log):
    txt = open(log, errors="ignore").read()
    m = re.findall(r"(\d+)%\|.*?\| (\d+)/(\d+)", txt)
    if m:
        pct, cur, tot = m[-1]
        print(f"progress: {pct}% ({cur}/{tot} files)")
print("PNG on disk:", len(glob.glob("/content/aerial-house-detection/datasets/whu/**/*.png", recursive=True)))

## Phase 3–4 — Dataset Conversion & Validation

Mask raster → instance bounding boxes via **8-connectivity connected components** (`cv2.connectedComponentsWithStats`), then normalize to YOLO format `class cx cy w h`.

**Validation rules (asserted, not assumed):**
1. Synthetic mask with known rectangles → bboxes must match exactly.
2. Reconstructed bbox must cover ≥95% of its mask component (IoU check).
3. Component area filter: drop < 64 px² (noise) — documented heuristic.
4. Count of labels per image logged; images with 0 components are kept (background samples) but listed.

**Known limitation:** touching/attached houses form ONE component → one bbox. This is the documented attached-house problem from the research phase; it inflates under-counting in dense row housing.

In [ ]:
# Phase 4 — Conversion module (mask -> YOLO bbox)
import os, cv2, numpy as np, json
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
SRC  = ROOT / "datasets/whu"
OUT  = ROOT / "datasets/whu_yolo"
MIN_AREA = 64  # px^2, drop specks (documented heuristic)

def mask_to_bboxes(mask: np.ndarray, min_area: int = MIN_AREA):
    """Binary mask -> list of (x, y, w, h) axis-aligned bboxes, 8-connectivity."""
    binary = (mask > 127).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, n):  # skip background label 0
        x, y, w, h, area = stats[i]
        if area >= min_area and w > 0 and h > 0:
            boxes.append((int(x), int(y), int(w), int(h)))
    return boxes

def boxes_to_yolo(boxes, W, H):
    """(x,y,w,h) px -> normalized 'cls cx cy w h' lines."""
    lines = []
    for x, y, w, h in boxes:
        lines.append(f"0 {(x + w/2)/W:.6f} {(y + h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
    return lines

assert mask_to_bboxes.__doc__ and boxes_to_yolo.__doc__
print("conversion funcs defined")

In [ ]:
# Phase 4 — UNIT TEST: geometric correctness on synthetic masks (must pass before conversion)
import numpy as np, cv2

# synthetic 100x100 mask with two known rectangles + one noise speck
m = np.zeros((100, 100), np.uint8)
m[10:30, 20:50] = 255      # box A: x=20 y=10 w=30 h=20
m[60:90, 70:95] = 255      # box B: x=70 y=60 w=25 h=30
m[95:97, 2:4]   = 255      # 4 px speck -> must be filtered (area < 64)

got = sorted(mask_to_bboxes(m), key=lambda b: b[0])
expected = [(20, 10, 30, 20), (70, 60, 25, 30)]
assert got == expected, f"bbox mismatch: got={got} expected={expected}"

def bbox_component_iou(bbox, mask):
    """IoU between bbox and the SINGLE component that bbox came from."""
    x, y, w, h = bbox
    n, labels, _, _ = cv2.connectedComponentsWithStats((mask > 127).astype(np.uint8), connectivity=8)
    lab = labels[y + h // 2, x + w // 2]          # component under bbox centre
    assert lab != 0, "bbox centre landed on background"
    gt = (labels == lab).astype(np.uint8)
    pred = np.zeros_like(gt); pred[y:y+h, x:x+w] = 1
    return np.logical_and(gt, pred).sum() / np.logical_or(gt, pred).sum()

for b in got:
    iou = bbox_component_iou(b, m)
    assert iou == 1.0, f"bbox {b} does not exactly bound its component: IoU={iou:.4f}"

lines = boxes_to_yolo(got, 100, 100)
assert len(lines) == 2 and all(l.startswith("0 ") and len(l.split()) == 5 for l in lines)
parts = [float(p) for p in lines[0].split()]
assert abs(parts[1] - (20 + 30/2)/100) < 1e-6

print("✅ ALL CONVERSION TESTS PASSED")
print("boxes:", got)
print("yolo :", lines)

## Phase 3 — Dataset Exploration (EDA)

Checks: image counts, annotation counts, objects/image, bbox size distribution (absolute + normalized), empty-image ratio, sample visualisation of ground truth.

In [ ]:
# Phase 3 — EDA on the SOURCE masks (before conversion)
import glob, os, cv2, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

SRC = Path("/content/aerial-house-detection/datasets/whu")
rows = []
for split in ["train", "val", "test"]:
    imgs = sorted((SRC / split / "Image").glob("*.png"))
    for ip in imgs:
        mp = SRC / split / "Mask" / ip.name
        if not mp.exists():
            rows.append({"split": split, "image": str(ip), "status": "MISSING_MASK"})
            continue
        m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        if m is None:
            rows.append({"split": split, "image": str(ip), "status": "CORRUPT_MASK"})
            continue
        n, _, stats, _ = cv2.connectedComponentsWithStats((m > 127).astype(np.uint8), connectivity=8)
        areas = [int(s[4]) for s in stats[1:] if s[4] >= 64]
        rows.append({"split": split, "image": str(ip), "status": "ok",
                     "h": m.shape[0], "w": m.shape[1], "n_boxes": len(areas),
                     "building_px_frac": float((m > 127).mean())})

df = pd.DataFrame(rows)
print("files seen:", len(df), "| corrupt/missing:", int((df.status != "ok").sum()))
ok = df[df.status == "ok"]
print(ok.groupby("split").agg(images=("image", "size"), boxes=("n_boxes", "sum"),
                               empty_images=("n_boxes", lambda s: int((s == 0).sum())),
                               median_boxes=("n_boxes", "median")))
print("\nbbox instance count per image:", ok.n_boxes.describe().round(2).to_dict())
print("building pixel fraction:", ok.building_px_frac.describe().round(3).to_dict())
print("image sizes:", ok.groupby(["w", "h"]).size().to_dict())
ok.to_csv("/content/aerial-house-detection/eda_summary.csv", index=False)

In [ ]:
# Phase 3 — EDA: bbox size distribution + GT visualisation (sampled, fast)
import cv2, numpy as np, random
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("/content/aerial-house-detection")
(ROOT / "outputs").mkdir(exist_ok=True)
SRC = ROOT / "datasets/whu"
random.seed(0)
sizes = []
for split in ["train", "val", "test"]:
    masks = sorted((SRC / split / "Mask").glob("*.png"))
    for mp in random.sample(masks, min(300, len(masks))):
        m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        n, _, stats, _ = cv2.connectedComponentsWithStats((m > 127).astype(np.uint8), 8)
        for s in stats[1:]:
            if s[4] >= 64:
                sizes.append((int(s[2]), int(s[3]), int(s[4]), (s[2]*s[3]) / m.size))
sizes = np.array(sizes)
print(f"sampled instances: {len(sizes)}")
for i, name in enumerate(["w_px", "h_px", "area_px", "area_frac"]):
    q = np.percentile(sizes[:, i], [10, 50, 90, 99])
    print(f"{name:10s} p10={q[0]:.2f} p50={q[1]:.2f} p90={q[2]:.2f} p99={q[3]:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.sqrt(sizes[:, 2]), bins=60); ax[0].set_title("sqrt(area) px"); ax[0].set_yscale("log")
ax[1].hist(sizes[:, 3] * 100, bins=60); ax[1].set_title("% of image area"); ax[1].set_yscale("log")
ax[2].scatter(sizes[:, 0], sizes[:, 1], s=2, alpha=0.2); ax[2].set_title("w vs h (px)")
plt.tight_layout(); plt.savefig(ROOT / "outputs/eda_bbox_sizes.png", dpi=110, bbox_inches="tight"); plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ip in zip(axes, random.sample(sorted((SRC / "train" / "Image").glob("*.png")), 3)):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    m = cv2.imread(str(SRC / "train" / "Mask" / ip.name), cv2.IMREAD_GRAYSCALE)
    n, _, stats, _ = cv2.connectedComponentsWithStats((m > 127).astype(np.uint8), 8)
    k = 0
    for s in stats[1:]:
        if s[4] < 64: continue
        x, y, w, h = s[:4]
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 40, 40), 2); k += 1
    ax.imshow(img); ax.set_title(f"{ip.name} — {k} boxes"); ax.axis("off")
plt.tight_layout(); plt.savefig(ROOT / "outputs/eda_gt_samples.png", dpi=110, bbox_inches="tight"); plt.close()
print("saved: outputs/eda_bbox_sizes.png, outputs/eda_gt_samples.png")

In [ ]:
# Phase 4 — Build YOLO dataset: convert all splits (symlinked images, real label txt files)
import os, cv2, numpy as np, json, time
from pathlib import Path
from collections import Counter

ROOT = Path("/content/aerial-house-detection")
SRC, OUT = ROOT / "datasets/whu", ROOT / "datasets/whu_yolo"
splits = ["train", "val", "test"]
stats_out, t0 = {}, time.time()

for split in splits:
    img_dir, lbl_dir = OUT / f"images/{split}", OUT / f"labels/{split}"
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    n_img = n_lbl = n_box = n_empty = n_bad = 0
    inst_per_img = []
    for ip in sorted((SRC / split / "Image").glob("*.png")):
        mp = SRC / split / "Mask" / ip.name
        im = cv2.imread(str(ip), cv2.IMREAD_COLOR)
        m  = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) if mp.exists() else None
        if im is None or m is None:
            n_bad += 1; continue
        H, W = m.shape
        boxes = mask_to_bboxes(m)
        lines = boxes_to_yolo(boxes, W, H)
        dst_img = img_dir / ip.name
        if not dst_img.exists():
            os.symlink(ip, dst_img)
        (lbl_dir / (ip.stem + ".txt")).write_text("\n".join(lines))
        n_img += 1; n_box += len(lines); n_empty += (len(lines) == 0)
    stats_out[split] = dict(images=n_img, boxes=n_box, empty_images=n_empty, corrupt=n_bad,
                            mean_boxes_per_img=round(n_box / max(n_img, 1), 2))
    print(split, stats_out[split])

(ROOT / "datasets/conversion_stats.json").write_text(json.dumps(stats_out, indent=2))
print("\nconversion time (s):", round(time.time() - t0, 1))

In [ ]:
# Phase 4 — Label integrity checks + dataset.yaml
import numpy as np
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
OUT = ROOT / "datasets/whu_yolo"
problems = []
for split in ["train", "val", "test"]:
    imgs = {p.stem for p in (OUT / f"images/{split}").glob("*.png")}
    lbls = {p.stem for p in (OUT / f"labels/{split}").glob("*.txt")}
    if imgs != lbls:
        problems.append(f"{split}: {len(imgs ^ lbls)} image/label mismatches")
    for lp in (OUT / f"labels/{split}").glob("*.txt"):
        for ln, line in enumerate(lp.read_text().splitlines()):
            v = line.split()
            if len(v) != 5: problems.append(f"{lp.name}:{ln} bad field count"); continue
            cls, *c = map(float, v)
            if cls != 0 or not all(0 <= x <= 1 for x in c):
                problems.append(f"{lp.name}:{ln} out-of-range {line}")
            if c[2] <= 0 or c[3] <= 0:
                problems.append(f"{lp.name}:{ln} non-positive wh")

print("label problems:", len(problems))
if problems: print("\n".join(problems[:20]))

yaml = f"""path: {OUT}
train: images/train
val: images/val
test: images/test
names:
  0: building
"""
(ROOT / "configs").mkdir(exist_ok=True)
(ROOT / "configs/dataset.yaml").write_text(yaml)
print(yaml)
assert not problems, "fix label problems before training"

## Phase 5–6 — Baseline Training (YOLO11s)

Config (from research plan, unchanged):
- `yolo11s.pt` COCO-pretrained, imgsz=640, batch=16, epochs=30, patience=10
- seed=42, deterministic, **test split untouched during training**
- augmentasi aerial-friendly bawaan Ultralytics: hsv, fliplr=0.5, flipud=0.5, degrees, scale, mosaic
- checkpoint best+last ke `experiments/exp01_yolo11s_baseline`
- Training runs in a background thread; poll cell below.

In [ ]:
# Phase 5-6 — Launch baseline training (background thread)
import os, threading, time
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
TRAIN = ROOT / "training_baseline.py"
TRAIN.write_text('''
from ultralytics import YOLO
m = YOLO("yolo11s.pt")
r = m.train(
    data="/content/aerial-house-detection/configs/dataset.yaml",
    imgsz=640, batch=16, epochs=30, patience=10,
    seed=42, deterministic=True, workers=2,
    project="/content/aerial-house-detection/experiments",
    name="exp01_yolo11s_baseline", exist_ok=True,
    plots=True, val=True, verbose=True,
)
print("BEST:", r.save_dir)
''')

def sh(cmd):
    import subprocess
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

if os.path.exists("/content/train_done.flag"): os.remove("/content/train_done.flag")
def run():
    r = sh("python /content/aerial-house-detection/training_baseline.py > /content/train.log 2>&1")
    open("/content/train_done.flag", "w").write("ok" if r.returncode == 0 else f"FAIL {r.returncode}")
threading.Thread(target=run, daemon=True).start()
print("Training launched -> /content/train.log")

In [ ]:
# Phase 6 — Poll training (re-run until FLAG appears; last epoch metrics shown)
import os, re
p = "/content/train.log"
if os.path.exists("/content/train_done.flag"):
    print("FLAG:", open("/content/train_done.flag").read())
if os.path.exists(p):
    txt = open(p, errors="ignore").read()
    eps = re.findall(r"^\s*(\d+)/30\s+.*$", txt, re.M)
    print("lines:", len(txt.splitlines()), "| epochs seen:", eps[-1] if eps else "starting")
    print(txt[-1200:])

## Phase 7 — Evaluation on the untouched TEST split

Metrics from Ultralytics `val`: Precision, Recall, mAP@0.5, mAP@0.5:0.95, per-class AP, inference time.
Custom analysis (the project differentiator from research): **house-count error (MAE / % error)** + TP/FP/FN/duplicate matching at IoU 0.5, and small-object (<32 px) recall.

In [ ]:
# Phase 7 — Standard metrics on test split (Ultralytics val)
from ultralytics import YOLO
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
best = ROOT / "experiments/exp01_yolo11s_baseline/weights/best.pt"
assert best.exists(), "train first"
model = YOLO(str(best))

m = model.val(data=str(ROOT / "configs/dataset.yaml"), split="test",
              imgsz=640, conf=0.25, iou=0.6, plots=True,
              project=str(ROOT / "experiments"), name="eval_test", exist_ok=True)
print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"F1        : {2*m.box.mp*m.box.mr/(m.box.mp+m.box.mr+1e-9):.4f}")
print(f"mAP@0.5   : {m.box.map50:.4f}")
print(f"mAP@.5:.95: {m.box.map:.4f}")
print(f"speed/ms  : pre={m.speed['preprocess']:.1f} inf={m.speed['inference']:.1f} loss={m.speed['loss']:.1f}")
import json
(ROOT / "experiments/eval_test/metrics.json").write_text(json.dumps({
    "precision": m.box.mp, "recall": m.box.mr,
    "f1": 2*m.box.mp*m.box.mr/(m.box.mp+m.box.mr+1e-9),
    "map50": m.box.map50, "map": m.box.map,
    "inference_ms": m.speed["inference"], "n_images": len(m.speed) and None,
}, indent=2))
print("saved metrics.json")

In [ ]:
# Phase 7 — Counting-error + error analysis (TP/FP/FN, duplicates, small-object recall)
import numpy as np, cv2, json
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
OUT = ROOT / "datasets/whu_yolo"

def iou(a, b):
    x1, y1, x2, y2 = a; X1, Y1, X2, Y2 = b
    iw = max(0, min(x2, X2) - max(x1, X1)); ih = max(0, min(y2, Y2) - max(y1, Y1))
    inter = iw * ih
    return inter / ((x2-x1)*(y2-y1) + (X2-X1)*(Y2-Y1) - inter + 1e-9)

def load_gt(txt, W, H):
    out = []
    for line in Path(txt).read_text().splitlines():
        _, cx, cy, w, h = map(float, line.split())
        out.append(((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H))
    return out

test_imgs = sorted((OUT / "images/test").glob("*.png"))
stats = {"tp": 0, "fp": 0, "fn": 0, "dup": 0, "small_tp": 0, "small_total": 0}
count_errs, per_img = [], []

for ip in test_imgs:
    img = cv2.imread(str(ip)); H, W = img.shape[:2]
    gt = load_gt(OUT / "labels/test" / (ip.stem + ".txt"), W, H)
    r = model.predict(str(ip), imgsz=640, conf=0.25, iou=0.5, verbose=False)[0]
    pr = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.empty((0, 4))

    matched = set()
    for b in pr:
        best_iou, best_j = 0, -1
        for j, g in enumerate(gt):
            v = iou(b, g)
            if v > best_iou: best_iou, best_j = v, j
        if best_iou >= 0.5:
            if best_j in matched: stats["dup"] += 1
            else: matched.add(best_j); stats["tp"] += 1
        else:
            stats["fp"] += 1
    stats["fn"] += len(gt) - len(matched)
    for g in gt:
        if (g[2]-g[0]) < 32 or (g[3]-g[1]) < 32:
            stats["small_total"] += 1
            if any(iou(b, g) >= 0.5 for b in pr): stats["small_tp"] += 1
    count_errs.append(abs(len(pr) - len(gt)))
    per_img.append({"image": ip.name, "n_gt": len(gt), "n_pred": len(pr),
                    "err": len(pr) - len(gt)})

count_errs = np.array(count_errs)
res = dict(stats)
res["count_mae"] = float(count_errs.mean())
res["count_mae_pct"] = float((count_errs / np.array([p["n_gt"] for p in per_img]).clip(1)).mean() * 100)
res["small_object_recall"] = stats["small_tp"] / max(stats["small_total"], 1)
res["precision"] = stats["tp"] / max(stats["tp"] + stats["fp"], 1)
res["recall"] = stats["tp"] / max(stats["tp"] + stats["fn"], 1)
res["f1"] = 2*res["precision"]*res["recall"] / max(res["precision"]+res["recall"], 1e-9)

print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()}, indent=2))
(ROOT / "experiments/eval_test/counting_analysis.json").write_text(json.dumps(res, indent=2))
import pandas as pd
pd.DataFrame(per_img).to_csv(ROOT / "experiments/eval_test/per_image_counts.csv", index=False)
torch_free = None

## Phase 8–10 — High-Resolution Inference (SAHI) + House Counting

Test design: mosaic 2×2 and 4×4 test tiles into 1024² and 2048² "orthophotos" with known GT boxes. Then compare:
1. **Direct inference** (whole image resized to 640, the naive approach)
2. **SAHI sliced inference** (slice 512, overlap 0.2, NMM merge)

Metric: matched detections at IoU 0.5 + count error. This is the experiment that justifies the slicing decision from the research phase.

In [ ]:
# Phase 8-10 — Build synthetic high-res mosaics from test tiles (no extra download)
import cv2, numpy as np, random
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
OUT = ROOT / "datasets/whu_yolo"
random.seed(42)

def build_mosaic(k: int, name: str):
    """Assemble k x k test tiles into one image + GT boxes in mosaic coords."""
    tiles = random.sample(sorted((OUT / "images/test").glob("*.png")), k * k)
    tile = cv2.imread(str(tiles[0])); h, w = tile.shape[:2]
    canvas = np.zeros((k * h, k * w, 3), np.uint8); gts = []
    for idx, tp in enumerate(tiles):
        r, c = divmod(idx, k)
        img = cv2.imread(str(tp)); canvas[r*h:(r+1)*h, c*w:(c+1)*w] = img
        for line in (OUT / "labels/test" / (tp.stem + ".txt")).read_text().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            gts.append(((c + cx - bw/2) * w, (r + cy - bh/2) * h,
                        (c + cx + bw/2) * w, (r + cy + bh/2) * h))
    cv2.imwrite(str(ROOT / f"outputs/{name}.png"), canvas) if (ROOT / "outputs").exists() else None
    return canvas, gts

(ROOT / "outputs").mkdir(exist_ok=True)
big2, gt2 = build_mosaic(2, "mosaic_1024")
big4, gt4 = build_mosaic(4, "mosaic_2048")
cv2.imwrite(str(ROOT / "outputs/mosaic_1024.png"), big2)
cv2.imwrite(str(ROOT / "outputs/mosaic_2048.png"), big4)
print("mosaic 1024:", big2.shape, "| GT boxes:", len(gt2))
print("mosaic 2048:", big4.shape, "| GT boxes:", len(gt4))

In [ ]:
# Phase 8-10 — Direct vs SAHI sliced inference on high-res mosaics
import numpy as np, json
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")

def iou_xyxy(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def match(preds, gts, thr=0.5):
    used = set(); tp = 0
    for p in preds:
        best, j = 0, -1
        for i, g in enumerate(gts):
            v = iou_xyxy(p, g)
            if v > best: best, j = v, i
        if best >= thr and j not in used: used.add(j); tp += 1
    return tp, len(preds) - tp, len(gts) - tp

def direct_predict(model, img, imgsz=640, conf=0.25):
    r = model.predict(img, imgsz=imgsz, conf=conf, iou=0.5, verbose=False)[0]
    return r.boxes.xyxy.cpu().numpy().tolist()

results = {}
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

det = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(ROOT / "experiments/exp01_yolo11s_baseline/weights/best.pt"),
    confidence_threshold=0.25, device="cuda:0",
)

for name, img, gts in [("mosaic_1024", big2, gt2), ("mosaic_2048", big4, gt4)]:
    d_preds = direct_predict(model, img)
    d_tp, d_fp, d_fn = match(d_preds, gts)
    entry = {"n_gt": len(gts),
             "direct": {"pred": len(d_preds), "tp": d_tp, "fp": d_fp, "fn": d_fn,
                        "count_err": abs(len(d_preds) - len(gts)),
                        "f1": 2*d_tp / max(2*d_tp + d_fp + d_fn, 1)}}
    for ss in (512, 640):
        s = get_sliced_prediction(img, det, slice_height=ss, slice_width=ss,
                                  overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                  postprocess_match_threshold=0.5, verbose=0)
        s_preds = [b.bbox.to_xyxy() for b in s.object_prediction_list]
        s_tp, s_fp, s_fn = match(s_preds, gts)
        entry[f"sahi_{ss}"] = {"pred": len(s_preds), "tp": s_tp, "fp": s_fp, "fn": s_fn,
                               "count_err": abs(len(s_preds) - len(gts)),
                               "f1": 2*s_tp / max(2*s_tp + s_fp + s_fn, 1)}
    results[name] = entry
    print(name, json.dumps(entry, indent=2))

(ROOT / "experiments/sahi_comparison.json").write_text(json.dumps(results, indent=2))

In [ ]:
# Phase 10 — House counting module (unique detections after merge)
import cv2, numpy as np, json
from pathlib import Path

def count_houses(image_path, model=None, detector=None,
                 conf=0.25, iou=0.5, use_sahi=True,
                 slice_size=512, overlap=0.2):
    """Return dict: count, boxes (xyxy original coords), scores, annotated image (BGR)."""
    img = cv2.imread(str(image_path))
    assert img is not None, f"cannot read {image_path}"
    if use_sahi:
        from sahi.predict import get_sliced_prediction
        s = get_sliced_prediction(img, detector, slice_height=slice_size, slice_width=slice_size,
                                  overlap_height_ratio=overlap, overlap_width_ratio=overlap,
                                  postprocess_match_threshold=iou, verbose=0)
        boxes = [b.bbox.to_xyxy() for b in s.object_prediction_list]
        scores = [float(b.score.value) for b in s.object_prediction_list]
    else:
        r = model.predict(img, imgsz=640, conf=conf, iou=iou, verbose=False)[0]
        boxes = r.boxes.xyxy.cpu().numpy().tolist()
        scores = r.boxes.conf.cpu().numpy().tolist()
    annotated = img.copy()
    for (x1, y1, x2, y2), sc in zip(boxes, scores):
        cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), (0, 220, 0), 2)
        cv2.putText(annotated, f"{sc:.2f}", (int(x1), max(int(y1) - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 220, 0), 1, cv2.LINE_AA)
    cv2.putText(annotated, f"Houses: {len(boxes)}", (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2, cv2.LINE_AA)
    return {"count": len(boxes), "boxes": boxes, "scores": scores, "annotated": annotated}

# demo (runs after training)
demo = count_houses(ROOT / "outputs/mosaic_1024.png", model=model, detector=det, use_sahi=True)
cv2.imwrite(str(ROOT / "outputs/mosaic_1024_counted.png"), demo["annotated"])
print("counted houses (1024 mosaic, SAHI):", demo["count"], "| GT:", len(gt2))
print("saved:", ROOT / "outputs/mosaic_1024_counted.png")

## Phase 11 — Video Detection (secondary, pipeline validation only)

No public UAV house video dataset exists in the research findings (VisDrone/UAVDT contain vehicles only). So this phase implements and validates the pipeline mechanics, not a house-count benchmark:

1. frame extraction → 2. detect per frame → 3. track with **BoT-SORT** (camera-motion compensation) → 4. count **unique track IDs**, never the sum of per-frame boxes → 5. write annotated video.

Validation: synthetic panning video built from `mosaic_2048` (known GT house count). Track-ID count is compared against GT; fragmentation is reported honestly.

In [ ]:
# Phase 11 — Video module + synthetic panning validation
import cv2, numpy as np, json
from pathlib import Path

def frame_detections(video_path, model, conf=0.25, stride=1):
    cap = cv2.VideoCapture(str(video_path)); frames = []
    i = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        if i % stride == 0: frames.append(fr)
        i += 1
    cap.release(); return frames

def count_unique_houses_video(video_path, model, tracker="botsort.yaml", conf=0.25):
    """Unique track IDs over the clip. Requires persist=True for cross-frame identity."""
    cap = cv2.VideoCapture(str(video_path))
    ids_seen, per_frame = set(), []
    while True:
        ok, fr = cap.read()
        if not ok: break
        r = model.track(fr, persist=True, tracker=tracker, conf=conf, verbose=False)[0]
        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().cpu().tolist()
            ids_seen.update(ids); per_frame.append(len(ids))
        else:
            per_frame.append(0)
    cap.release()
    return {"unique_ids": len(ids_seen), "frames": len(per_frame),
            "sum_per_frame_detections": int(sum(per_frame))}

# build synthetic panning video from mosaic_2048 (window 640, 40 frames, step 20 px)
WIN, N, STEP = 640, 40, 18
H4, W4 = big4.shape[:2]
vw = cv2.VideoWriter(str(ROOT / "outputs/synthetic_pan.mp4"),
                     cv2.VideoWriter_fourcc(*"mp4v"), 10, (WIN, WIN))
for i in range(N):
    x = min(i * STEP, W4 - WIN)
    vw.write(big4[300:300+WIN, x:x+WIN])
vw.release()
print("video written:", ROOT / "outputs/synthetic_pan.mp4", "| GT houses in mosaic:", len(gt4))

vid = count_unique_houses_video(ROOT / "outputs/synthetic_pan.mp4", model, tracker="botsort.yaml")
vid["gt_houses_in_scene"] = len(gt4)
vid["note"] = "synthetic panning clip; pipeline validation only, not a UAV benchmark"
print(json.dumps(vid, indent=2))
(ROOT / "experiments/video_validation.json").write_text(json.dumps(vid, indent=2))

## Phase 12 — Experiment Summary & Artifacts

Consolidated table: config, metrics, counting error, SAHI comparison, video validation. Saved as JSON/CSV for the repository report.

In [ ]:
# Phase 12 — Consolidated experiment summary
import json, glob, os
from pathlib import Path
import pandas as pd

ROOT = Path("/content/aerial-house-detection")
rows = []

mt = json.loads((ROOT / "experiments/eval_test/metrics.json").read_text()) if (ROOT / "experiments/eval_test/metrics.json").exists() else {}
ct = json.loads((ROOT / "experiments/eval_test/counting_analysis.json").read_text()) if (ROOT / "experiments/eval_test/counting_analysis.json").exists() else {}
if mt:
    rows.append({"exp": "exp01_yolo11s_baseline", "set": "test",
                 "precision": round(mt.get("precision", 0), 4), "recall": round(mt.get("recall", 0), 4),
                 "f1": round(mt.get("f1", 0), 4), "map50": round(mt.get("map50", 0), 4),
                 "map50_95": round(mt.get("map", 0), 4), "count_mae": round(ct.get("count_mae", 0), 2),
                 "count_mae_pct": round(ct.get("count_mae_pct", 0), 2),
                 "small_recall": round(ct.get("small_object_recall", 0), 4)})
# SAHI comparison as extra rows
sp = ROOT / "experiments/sahi_comparison.json"
if sp.exists():
    s = json.loads(sp.read_text())
    for mos, v in s.items():
        for mode in [k for k in v if k == "direct" or k.startswith("sahi")]:
            rows.append({"exp": f"{mos}_{mode}", "set": "mosaic", "precision": None, "recall": None,
                         "f1": round(v[mode]["f1"], 4), "map50": None, "map50_95": None,
                         "count_mae": v[mode]["count_err"], "count_mae_pct": None, "small_recall": None})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(ROOT / "experiments/experiment_summary.csv", index=False)

print("\nArtifacts:")
for p in sorted(glob.glob(str(ROOT / "experiments/**/*"), recursive=True)):
    if os.path.isfile(p) and p.endswith((".json", ".csv", ".png", ".pt", ".yaml")):
        print(" ", p.replace(str(ROOT) + "/", ""), round(os.path.getsize(p) / 1e6, 2), "MB")

## Training History — per-epoch curves

Loss components, precision, recall, mAP@0.5, mAP@0.5:0.95 and learning rate from Ultralytics `results.csv`. Plots are saved next to the checkpoint.

In [ ]:
# Training history — per-epoch curves (not just final values)
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/aerial-house-detection")
run = ROOT / "experiments/exp01_yolo11s_baseline"
csv = run / "results.csv"
df = pd.read_csv(csv)
df.columns = [c.strip() for c in df.columns]
print(df.tail(3).to_string(index=False))

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
for c in [c for c in df.columns if "loss" in c]:
    ax[0, 0].plot(df["epoch"], df[c], label=c)
ax[0, 0].set_title("losses"); ax[0, 0].legend(fontsize=7); ax[0, 0].set_xlabel("epoch")
for c in ["metrics/precision(B)", "metrics/recall(B)"]:
    if c in df: ax[0, 1].plot(df["epoch"], df[c], label=c)
ax[0, 1].set_title("precision / recall"); ax[0, 1].legend(); ax[0, 1].set_xlabel("epoch")
for c in ["metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
    if c in df: ax[1, 0].plot(df["epoch"], df[c], label=c)
ax[1, 0].set_title("mAP"); ax[1, 0].legend(); ax[1, 0].set_xlabel("epoch")
lr_cols = [c for c in df.columns if "lr/" in c]
for c in lr_cols[:3]: ax[1, 1].plot(df["epoch"], df[c], label=c)
ax[1, 1].set_title("learning rate"); ax[1, 1].legend(fontsize=7); ax[1, 1].set_xlabel("epoch")
for a in ax.flat: a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(run / "training_history.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved:", run / "training_history.png")

In [ ]:
# wait helper: block up to ~100s then report (safe to re-run)
import time, glob, os, re
t0 = time.time()
while time.time() - t0 < 100:
    if os.path.exists("/content/dl_done.flag"):
        break
    time.sleep(10)
if os.path.exists("/content/dl_done.flag"):
    print("FLAG:", open("/content/dl_done.flag").read())
txt = open("/content/dl_whu.log", errors="ignore").read()
m = re.findall(r"(\d+)%\|.*?\| (\d+)/(\d+)", txt)
if m: print(f"progress: {m[-1][0]}% ({m[-1][1]}/{m[-1][2]})")
print("PNG on disk:", len(glob.glob("/content/aerial-house-detection/datasets/whu/**/*.png", recursive=True)))

In [ ]:
# Phase 7 — Qualitative error analysis: GT | prediction | overlay
import cv2, numpy as np, random
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("/content/aerial-house-detection")
OUT = ROOT / "datasets/whu_yolo"
random.seed(7)

# pick tiles with many houses (dense) and a few with small ones
cands = sorted((OUT / "images/test").glob("*.png"))
counts = []
for ip in cands[:400]:
    n = len((OUT / "labels/test" / (ip.stem + ".txt")).read_text().splitlines())
    counts.append((n, ip))
dense = [ip for n, ip in sorted(counts, reverse=True)[:2]]
mixed = [ip for n, ip in counts if 8 <= n <= 15][:1]
picks = dense + mixed

fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4.2 * len(picks)))
for row, ip in zip(axes, picks):
    img0 = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    h, w = img0.shape[:2]
    gt_boxes, gt_img = [], img0.copy()
    for line in (OUT / "labels/test" / (ip.stem + ".txt")).read_text().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = (cx-bw/2)*w, (cy-bh/2)*h; x2, y2 = (cx+bw/2)*w, (cy+bh/2)*h
        gt_boxes.append((x1, y1, x2, y2))
        cv2.rectangle(gt_img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 40, 40), 2)
    r = model.predict(str(ip), imgsz=640, conf=0.25, iou=0.5, verbose=False)[0]
    pr_boxes = r.boxes.xyxy.cpu().numpy().tolist() if r.boxes is not None else []
    pr_img, both = img0.copy(), img0.copy()
    for b in pr_boxes:
        cv2.rectangle(pr_img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 220, 0), 2)
    for b in gt_boxes:
        cv2.rectangle(both, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 40, 40), 2)
    for b in pr_boxes:
        cv2.rectangle(both, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 220, 0), 1)
    for ax, im, t in zip(row, [gt_img, pr_img, both],
                         [f"GT: {len(gt_boxes)}", f"Pred: {len(pr_boxes)}", "GT (red) vs Pred (green)"]):
        ax.imshow(im); ax.set_title(f"{ip.name} — {t}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(ROOT / "outputs/eval_gt_vs_pred.png", dpi=110, bbox_inches="tight"); plt.close()
print("saved: outputs/eval_gt_vs_pred.png | tiles:", [p.name for p in picks])

In [ ]:
# Deliverable — verify, export and download the trained model
import hashlib, shutil, zipfile, json
from pathlib import Path
from ultralytics import YOLO

ROOT = Path("/content/aerial-house-detection")
best = ROOT / "experiments/exp01_yolo11s_baseline/weights/best.pt"
assert best.exists(), "best.pt missing"

sha = hashlib.sha256(best.read_bytes()).hexdigest()
print("weights:", best, round(best.stat().st_size/1e6, 2), "MB")
print("sha256 :", sha)

m = YOLO(str(best))
print("params :", sum(p.numel() for p in m.model.parameters()))

# sanity prediction on one test tile
import cv2
tile = sorted((ROOT / "datasets/whu_yolo/images/test").glob("*.png"))[0]
r = m.predict(str(tile), imgsz=640, conf=0.25, verbose=False)[0]
print("sanity :", tile.name, "detections =", 0 if r.boxes is None else len(r.boxes))

# export deployable format
onnx = m.export(format="onnx", imgsz=640, dynamic=False, simplify=True)
print("onnx   :", onnx, round(Path(onnx).stat().st_size/1e6, 2), "MB")

In [ ]:
# proof on a non-empty tile + package deliverables + browser download
import json, zipfile, cv2
from pathlib import Path
from ultralytics import YOLO

ROOT = Path("/content/aerial-house-detection")
OUT = ROOT / "datasets/whu_yolo"
m = YOLO(str(ROOT / "experiments/exp01_yolo11s_baseline/weights/best.pt"))

# first tile with >= 5 GT boxes
tile = next(p for p in sorted((OUT / "images/test").glob("*.png"))
            if len((OUT / "labels/test" / (p.stem + ".txt")).read_text().splitlines()) >= 5)
r = m.predict(str(tile), imgsz=640, conf=0.25, verbose=False)[0]
n_pred = 0 if r.boxes is None else len(r.boxes)
n_gt = len((OUT / "labels/test" / (tile.stem + ".txt")).read_text().splitlines())
print("proof:", tile.name, "| GT:", n_gt, "| predicted:", n_pred)

bundle = ROOT / "outputs/aerial_house_detection_model.zip"
files = [
    "experiments/exp01_yolo11s_baseline/weights/best.pt",
    "experiments/exp01_yolo11s_baseline/weights/best.onnx",
    "experiments/exp01_yolo11s_baseline/results.csv",
    "experiments/exp01_yolo11s_baseline/training_history.png",
    "experiments/eval_test/metrics.json",
    "experiments/eval_test/counting_analysis.json",
    "experiments/eval_test/confusion_matrix.png",
    "experiments/eval_test/BoxPR_curve.png",
    "experiments/sahi_comparison.json",
    "experiments/video_validation.json",
    "experiments/experiment_summary.csv",
    "outputs/eval_gt_vs_pred.png",
    "outputs/mosaic_1024_counted.png",
]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p = ROOT / f
        if p.exists():
            z.write(p, f)
print("bundle:", bundle, round(bundle.stat().st_size / 1e6, 2), "MB")

from google.colab import files
files.download(str(bundle))
print("download triggered")

In [ ]:
# quick health check (fast)
print("alive")

In [ ]:
# training poll (waits up to 200s)
import time, os, re
t0 = time.time()
while time.time() - t0 < 200:
    if os.path.exists("/content/train_done.flag"): break
    time.sleep(15)
if os.path.exists("/content/train_done.flag"):
    print("FLAG:", open("/content/train_done.flag").read())
txt = open("/content/train.log", errors="ignore").read() if os.path.exists("/content/train.log") else ""
epochs = re.findall(r"\s(\d+)/30\s", txt)
print("last epoch marker:", (epochs or ["none"])[-1])
tail = [l.split("\r")[-1] for l in txt.splitlines() if l.strip()][-2:]
print("\n".join(tail)[-400:])